# Lista 3 — Sistemas de Equações Lineares

Notebook pronto para rodar no **Jupyter**, com:
- enunciados reescritos;
- contexto das funções utilizadas;
- resolução em Python;
- comentários sobre convergência, fatorações e interpretação física quando necessário.

In [6]:
import numpy as np
import sympy as sp

np.set_printoptions(precision=8, suppress=True)

## Contexto das funções utilizadas

As rotinas abaixo foram implementadas de forma **genérica**:

- `forward_substitution(L, b)` e `back_substitution(U, b)`;
- `gaussian_elimination_pp(A, b)`: eliminação de Gauss com pivotamento parcial;
- `lu_doolittle_pp(A)`: fatoração LU com pivotamento parcial;
- `crout_decomposition(A)`: decomposição de Crout;
- `jacobi(A, b)`, `gauss_seidel(A, b)`, `sor(A, b, w)`;
- funções auxiliares para:
  - norma infinita,
  - resíduo,
  - erro real,
  - verificação de dominância diagonal.

Essas funções podem ser reaproveitadas em outros exercícios.

In [7]:
def forward_substitution(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    y = np.zeros(n, dtype=float)
    for i in range(n):
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]
    return y

def back_substitution(U, b):
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n, dtype=float)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i]
    return x

def gaussian_elimination_pp(A, b):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)
    n = len(b)
    for k in range(n - 1):
        pivot = k + np.argmax(np.abs(A[k:, k]))
        if np.isclose(A[pivot, k], 0):
            raise ValueError("Matriz singular.")
        if pivot != k:
            A[[k, pivot]] = A[[pivot, k]]
            b[[k, pivot]] = b[[pivot, k]]
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            b[i] -= m * b[k]
    return back_substitution(A, b), A, b

def lu_doolittle_pp(A):
    A = np.array(A, dtype=float, copy=True)
    n = A.shape[0]
    P = np.eye(n)
    L = np.zeros((n, n), dtype=float)
    U = A.copy()

    for k in range(n):
        pivot = k + np.argmax(np.abs(U[k:, k]))
        if np.isclose(U[pivot, k], 0):
            raise ValueError("Matriz singular.")
        if pivot != k:
            U[[k, pivot]] = U[[pivot, k]]
            P[[k, pivot]] = P[[pivot, k]]
            if k > 0:
                L[[k, pivot], :k] = L[[pivot, k], :k]
        L[k, k] = 1.0
        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]
    return P, L, U

def solve_lu(P, L, U, b):
    Pb = P @ np.asarray(b, dtype=float)
    y = forward_substitution(L, Pb)
    x = back_substitution(U, y)
    return x

def crout_decomposition(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.zeros((n, n), dtype=float)
    U = np.eye(n, dtype=float)

    for j in range(n):
        for i in range(j, n):
            L[i, j] = A[i, j] - np.sum(L[i, :j] * U[:j, j])
        for i in range(j + 1, n):
            U[j, i] = (A[j, i] - np.sum(L[j, :j] * U[:j, i])) / L[j, j]
    return L, U

def jacobi(A, b, x0=None, tol=1e-6, maxit=10000):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n) if x0 is None else np.asarray(x0, dtype=float).copy()
    D = np.diag(A)
    R = A - np.diagflat(D)
    hist = [x.copy()]
    for k in range(1, maxit + 1):
        x_new = (b - R @ x) / D
        hist.append(x_new.copy())
        err = np.linalg.norm(x_new - x, np.inf) / max(np.linalg.norm(x_new, np.inf), 1.0)
        if err < tol:
            return x_new, k, np.array(hist)
        x = x_new
    return x, maxit, np.array(hist)

def gauss_seidel(A, b, x0=None, tol=1e-6, maxit=10000):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n) if x0 is None else np.asarray(x0, dtype=float).copy()
    hist = [x.copy()]
    for k in range(1, maxit + 1):
        x_old = x.copy()
        for i in range(n):
            x[i] = (b[i] - np.dot(A[i, :i], x[:i]) - np.dot(A[i, i+1:], x_old[i+1:])) / A[i, i]
        hist.append(x.copy())
        err = np.linalg.norm(x - x_old, np.inf) / max(np.linalg.norm(x, np.inf), 1.0)
        if err < tol:
            return x, k, np.array(hist)
    return x, maxit, np.array(hist)

def sor(A, b, w, x0=None, tol=1e-6, maxit=10000):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n) if x0 is None else np.asarray(x0, dtype=float).copy()
    hist = [x.copy()]
    for k in range(1, maxit + 1):
        x_old = x.copy()
        for i in range(n):
            gs_val = (b[i] - np.dot(A[i, :i], x[:i]) - np.dot(A[i, i+1:], x_old[i+1:])) / A[i, i]
            x[i] = (1 - w)*x_old[i] + w*gs_val
        hist.append(x.copy())
        err = np.linalg.norm(x - x_old, np.inf) / max(np.linalg.norm(x, np.inf), 1.0)
        if err < tol:
            return x, k, np.array(hist)
    return x, maxit, np.array(hist)

def is_strictly_diagonally_dominant(A):
    A = np.asarray(A, dtype=float)
    diag = np.abs(np.diag(A))
    off = np.sum(np.abs(A), axis=1) - diag
    return np.all(diag > off)

def inf_norm(v):
    return np.linalg.norm(np.asarray(v, dtype=float), np.inf)

## Questão 1

**Enunciado reescrito.** Resolver o sistema do circuito e responder:
1. LU pode ser aplicada?
2. Cholesky pode ser aplicada?
3. Resolver por eliminação de Gauss (na prática, também vou mostrar a LU correspondente).

In [8]:
A = np.array([
    [8, -4, -2],
    [-4, 6, -2],
    [-2, -2, 10]
], dtype=float)
b = np.array([10, 0, 4], dtype=float)

print("Matriz A:")
print(A)
print("\nVetor b:")
print(b)

x_gauss, Utri, btri = gaussian_elimination_pp(A, b)
P, L, U = lu_doolittle_pp(A)
x_lu = solve_lu(P, L, U, b)

print("\nSolução por Gauss:")
print(x_gauss)

print("\nFatoração LU com pivotamento parcial:")
print("P =\n", P)
print("L =\n", L)
print("U =\n", U)
print("Solução via LU =", x_lu)

print("\nJustificativas:")
print(f"- LU é aplicável porque det(A) = {np.linalg.det(A):.6f} ≠ 0.")
try:
    chol = np.linalg.cholesky(A)
    print("- Cholesky é aplicável porque A é simétrica positiva definida.")
    print("Fator de Cholesky L =\n", chol)
except np.linalg.LinAlgError:
    print("- Cholesky não é aplicável.")

Matriz A:
[[ 8. -4. -2.]
 [-4.  6. -2.]
 [-2. -2. 10.]]

Vetor b:
[10.  0.  4.]

Solução por Gauss:
[2.75862069 2.31034483 1.4137931 ]

Fatoração LU com pivotamento parcial:
P =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
L =
 [[ 1.    0.    0.  ]
 [-0.5   1.    0.  ]
 [-0.25 -0.75  1.  ]]
U =
 [[ 8.   -4.   -2.  ]
 [ 0.    4.   -3.  ]
 [ 0.    0.    7.25]]
Solução via LU = [2.75862069 2.31034483 1.4137931 ]

Justificativas:
- LU é aplicável porque det(A) = 232.000000 ≠ 0.
- Cholesky é aplicável porque A é simétrica positiva definida.
Fator de Cholesky L =
 [[ 2.82842712  0.          0.        ]
 [-1.41421356  2.          0.        ]
 [-0.70710678 -1.5         2.6925824 ]]


## Questão 2

**Enunciado reescrito.** Escrever o sistema da Figura 4.7 na forma \(Ax=b\), calcular \(A^{-1}\) e usar esse resultado para obter \(I_x\).

In [9]:
# Sistema lido do enunciado:
# 4(I1 + I2 + Ix) + 10 I1 = 100
# 4(I1 + I2 + Ix) + 3(I2 + Ix) + 10 Ix = 100 - 2 Ix
# 4(I1 + I2 + Ix) + 3(I2 + Ix) + 9 Ix = 100 - 2 Ix

A = np.array([
    [14, 4, 4],
    [4, 7, 19],
    [4, 7, 18]
], dtype=float)
b = np.array([100, 100, 100], dtype=float)

Ainv = np.linalg.inv(A)
sol = Ainv @ b
I1, I2, Ix = sol

print("Forma matricial:")
print("A =\n", A)
print("b =", b)

print("\nA^{-1} =")
print(Ainv)

print("\nSolução [I1, I2, Ix]^T =")
print(sol)

print(f"\nLogo, Ix = {Ix:.6f}")

Forma matricial:
A =
 [[14.  4.  4.]
 [ 4.  7. 19.]
 [ 4.  7. 18.]]
b = [100. 100. 100.]

A^{-1} =
[[ 0.08536585  0.53658537 -0.58536585]
 [-0.04878049 -2.87804878  3.04878049]
 [-0.          1.         -1.        ]]

Solução [I1, I2, Ix]^T =
[ 3.65853659 12.19512195  0.        ]

Logo, Ix = 0.000000


## Questão 3

**Enunciado reescrito.** Montar as equações nodais restantes e resolver o sistema para as tensões dos nós \(V_1, V_2, V_3, V_4\).

O enunciado já fornece a equação do nó 1:
\[
-4V_1 + 2V_2 + V_4 = -100.
\]
As demais são obtidas pela mesma Lei de Kirchhoff das Correntes.

In [10]:
A = np.array([
    [-4,  2,  0,  1],
    [ 1, -2,  1,  0],
    [ 0,  2, -3,  1],
    [ 5,  0,  5, -12]
], dtype=float)
b = np.array([-100, 0, 0, 0], dtype=float)

sol, _, _ = gaussian_elimination_pp(A, b)

print("Sistema Ax = b:")
print("A =\n", A)
print("b =", b)
print("\nSolução [V1, V2, V3, V4]^T =")
print(sol)

Sistema Ax = b:
A =
 [[ -4.   2.   0.   1.]
 [  1.  -2.   1.   0.]
 [  0.   2.  -3.   1.]
 [  5.   0.   5. -12.]]
b = [-100.    0.    0.    0.]

Solução [V1, V2, V3, V4]^T =
[76. 72. 68. 60.]


## Questão 4

**Enunciado reescrito.** Verificar se Gauss-Seidel possui convergência assegurada e, se sim, obter a solução com erro relativo menor que \(10^{-2}\).

In [11]:
A = np.array([
    [20, -10, -4],
    [-10, 20, -5],
    [-4, -5, 20]
], dtype=float)
b = np.array([26, 0, 7], dtype=float)

print("Dominância diagonal estrita:", is_strictly_diagonally_dominant(A))

x_gs, it_gs, hist = gauss_seidel(A, b, tol=1e-2)

print("\nComo A é estritamente diagonal dominante, a convergência do Gauss-Seidel fica assegurada.")
print(f"Iterações até critério pedido: {it_gs}")
print("Solução aproximada:")
print(x_gs)

x_exact = np.linalg.solve(A, b)
rel_err = inf_norm(x_exact - x_gs) / inf_norm(x_exact)
print(f"Erro relativo real (comparando com solução exata) = {rel_err:.6e}")

Dominância diagonal estrita: True

Como A é estritamente diagonal dominante, a convergência do Gauss-Seidel fica assegurada.
Iterações até critério pedido: 6
Solução aproximada:
[2.21418646 1.39022728 1.14039411]
Erro relativo real (comparando com solução exata) = 7.147159e-03


## Questão 5

**Enunciado reescrito.** Resolver o sistema por decomposição LU com pivotamento.

In [12]:
A = np.array([
    [ 2, -6, -1],
    [-3, -1,  7],
    [-8,  1, -2]
], dtype=float)
b = np.array([-38, -34, -20], dtype=float)

P, L, U = lu_doolittle_pp(A)
x = solve_lu(P, L, U, b)

print("P =\n", P)
print("L =\n", L)
print("U =\n", U)
print("\nSolução x =")
print(x)

P =
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]
L =
 [[ 1.          0.          0.        ]
 [-0.25        1.          0.        ]
 [ 0.375       0.23913043  1.        ]]
U =
 [[-8.          1.         -2.        ]
 [ 0.         -5.75       -1.5       ]
 [ 0.          0.          8.10869565]]

Solução x =
[ 4.  8. -2.]


## Questão 6

**Enunciado reescrito.** Efetuar a decomposição de Crout para o sistema dado.

In [13]:
A = np.array([
    [ 2, -6,  1],
    [-1,  7, -1],
    [ 1, -3,  2]
], dtype=float)
b = np.array([12, -8, 16], dtype=float)

L, U = crout_decomposition(A)
y = forward_substitution(L, b)
x = back_substitution(U, y)

print("L (Crout) =\n", L)
print("U (Crout) =\n", U)
print("\nVerificação de fatoração A ≈ L@U:")
print(L @ U)
print("\nSolução do sistema usando a decomposição:")
print(x)

L (Crout) =
 [[ 2.   0.   0. ]
 [-1.   4.   0. ]
 [ 1.   0.   1.5]]
U (Crout) =
 [[ 1.    -3.     0.5  ]
 [ 0.     1.    -0.125]
 [ 0.     0.     1.   ]]

Verificação de fatoração A ≈ L@U:
[[ 2. -6.  1.]
 [-1.  7. -1.]
 [ 1. -3.  2.]]

Solução do sistema usando a decomposição:
[3.66666667 0.33333333 6.66666667]


## Questão 7

**Enunciado reescrito.** Resolver o sistema dos reatores por:
- Gauss-Jacobi;
- Gauss-Seidel;
- sub-relaxamento \(\omega = 0{,}9\);
- sobre-relaxamento \(\omega = 1{,}2\);

com tolerância \(10^{-6}\).

In [14]:
A = np.array([
    [15, -3, -1],
    [-3, 18, -6],
    [-4, -1, 12]
], dtype=float)
b = np.array([3800, 1200, 2350], dtype=float)

x_j, it_j, _ = jacobi(A, b, tol=1e-6)
x_gs, it_gs, _ = gauss_seidel(A, b, tol=1e-6)
x_sor09, it_sor09, _ = sor(A, b, 0.9, tol=1e-6)
x_sor12, it_sor12, _ = sor(A, b, 1.2, tol=1e-6)

print("Jacobi        :", x_j,     "| iterações =", it_j)
print("Gauss-Seidel  :", x_gs,    "| iterações =", it_gs)
print("SOR w=0.9     :", x_sor09, "| iterações =", it_sor09)
print("SOR w=1.2     :", x_sor12, "| iterações =", it_sor12)

print("\nComentário:")
print("- Menor número de iterações indica convergência mais rápida para este problema.")

Jacobi        : [320.20713303 227.20189176 321.50244508] | iterações = 15
Gauss-Seidel  : [320.20720558 227.20203164 321.50257116] | iterações = 10
SOR w=0.9     : [320.20714527 227.20196415 321.50253082] | iterações = 13
SOR w=1.2     : [320.20721882 227.20210662 321.50256842] | iterações = 16

Comentário:
- Menor número de iterações indica convergência mais rápida para este problema.


## Questão 8

**Enunciado reescrito.** Para o sistema das correntes:
1. determinar erro real e resíduo;
2. calcular as normas infinitas da solução exata, do erro e do resíduo;
3. calcular o número de condição da matriz;
4. encontrar a solução por um método direto e um método iterativo.

**Escolhas adotadas:**
- método direto: **LU com pivotamento parcial**;
- método iterativo: **Gauss-Seidel** com tolerância \(10^{-6}\).

In [15]:
A = np.array([
    [ 8.0, -4.0,  0.0, -1.0,  0.0,  0.0],
    [-4.0, 11.5, -2.5,  0.0, -5.0,  0.0],
    [ 0.0, -2.5,  4.5,  0.0,  0.0, -2.0],
    [-1.0,  0.0,  0.0,  3.0, -2.0,  0.0],
    [ 0.0, -5.0,  0.0, -2.0,  8.5, -1.5],
    [ 0.0,  0.0, -2.0,  0.0, -1.5,  8.0]
], dtype=float)

b = np.array([20, -12, 14, 8, -30, 0], dtype=float)

# solução exata de referência por LU
P, L, U = lu_doolittle_pp(A)
x_direct = solve_lu(P, L, U, b)

# solução iterativa
x_iter, it_iter, _ = gauss_seidel(A, b, tol=1e-6)

erro_real = x_direct - x_iter
residuo = b - A @ x_iter

print("Solução exata (método direto):")
print(x_direct)

print("\nSolução iterativa (Gauss-Seidel):")
print(x_iter)
print("iterações =", it_iter)

print("\nErro real = x_exato - x_iterativo")
print(erro_real)

print("\nResíduo = b - A x_iterativo")
print(residuo)

print("\nNormas infinitas:")
print(f"||x_exato||_inf   = {inf_norm(x_direct):.12f}")
print(f"||erro||_inf      = {inf_norm(erro_real):.12e}")
print(f"||residuo||_inf   = {inf_norm(residuo):.12e}")

print("\nNúmeros de condição:")
print(f"cond_2(A)   = {np.linalg.cond(A):.12f}")
print(f"cond_inf(A) = {np.linalg.cond(A, np.inf):.12f}")

Solução exata (método direto):
[ 1.04696876 -2.75756949  1.2689151  -0.593972   -5.41444238 -0.69797917]

Solução iterativa (Gauss-Seidel):
[ 1.04698283 -2.75755201  1.2689284  -0.59395368 -5.41442636 -0.69797284]
iterações = 50

Erro real = x_exato - x_iterativo
[-0.00001408 -0.00001747 -0.0000133  -0.00001832 -0.00001602 -0.00000633]

Resíduo = b - A x_iterativo
[-0.00002439 -0.00003133 -0.0000035  -0.00000886 -0.00000262  0.        ]

Normas infinitas:
||x_exato||_inf   = 5.414442375336
||erro||_inf      = 1.832096552734e-05
||residuo||_inf   = 3.133157618862e-05

Números de condição:
cond_2(A)   = 21.587851752875
cond_inf(A) = 35.951743798423
